In [11]:
library(data.table)
library(readr)
library(stringr)
library(lubridate) # For handling timestamps
for (year_to_process in year_range) {
  source("~/pids-drg-claims/data-cleaning/00a-parameters.r")
  print(year_to_process)
  for (to_sample in c(TRUE, FALSE)) {
    list_of_divisors <- if (to_sample) c(625, 125, 25, 5) else c(1)
    for (sample_size_divisor in list_of_divisors) { # Load necessary parameter files
      suffix <- paste0(ifelse(exists("to_sample") && to_sample, paste0("_sampled_", sample_size_divisor, "_"), "_full_"))

      # Validate suffix
      valid_suffix_pattern <- "_full_|_sampled_\\d+_"
      if (!grepl(valid_suffix_pattern, suffix)) {
        stop("Invalid suffix: ", suffix, ". Expected '_full_' or '_sampled_<sample_size_divisor>_'")
      }

      # Read and filter filenames
      files <- list.files(
        path = "~/pids-drg-claims/data-cleaning/data/chkpts/chkpt_12_mapping/",
        pattern = paste0("map_", year_to_process, suffix, ".*fork_\\d+_part_\\d+\\.rds"),
        full.names = TRUE
      )

      if (length(files) == 0) {
        message("No files found for the specified year_to_load and suffix: ", suffix)
        next
      }

      # Function to parse file names
      parse_filename <- function(filename) {
        pattern <- paste0("map_(\\d{4})", suffix, "(\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\.\\d+)_fork_\\d+_part_\\d+\\.rds")
        matches <- str_match(filename, pattern)

        if (!is.na(matches[1])) {
          return(list(
            year_to_load = as.integer(matches[2]),
            timestamp = matches[3],
            filename = filename
          ))
        } else {
          return(NULL)
        }
      }

      parsed_files <- lapply(files, parse_filename)
      parsed_files <- Filter(Negate(is.null), parsed_files)

      if (length(parsed_files) == 0) {
        message("No valid parsed files found for year_to_load: ", year_to_process, " and suffix: ", suffix)
        next
      }

      # Select latest timestamp for the given year_to_load
      latest_timestamp <- max(sapply(parsed_files, `[[`, "timestamp"))
      latest_files <- Filter(function(x) x$timestamp == latest_timestamp, parsed_files)

      # Define output directories
      # Define output directories based on sampling flag
      mapping_type <- if (to_sample) "sampled" else "full"
      partial_output_folder <- paste0("~/pids-drg-claims/data-cleaning/data/chkpts/chkpt_12_mapping/", mapping_type, "_rds_files/", year_to_process, suffix, latest_timestamp, "/")
      final_output_folder <- paste0("~/pids-drg-claims/data-cleaning/data/chkpts/chkpt_12_mapping/", mapping_type, "_final_mappings/", year_to_process, suffix, latest_timestamp, "/")

      dir.create(final_output_folder, recursive = TRUE, showWarnings = FALSE)
      dir.create(partial_output_folder, recursive = TRUE, showWarnings = FALSE)

      # Read and combine mappings
      final_icd_dt <- rbindlist(lapply(latest_files, function(file_info) {
        mappings_list <- readRDS(file_info$filename)
        return(mappings_list$icd_mappings)
      }), use.names = TRUE, fill = TRUE)

      final_rvs_dt <- rbindlist(lapply(latest_files, function(file_info) {
        mappings_list <- readRDS(file_info$filename)
        return(mappings_list$rvs_mappings)
      }), use.names = TRUE, fill = TRUE)

      # Deduplicate while allowing multiple mappings per source code
      final_icd_dt <- unique(final_icd_dt)
      final_rvs_dt <- unique(final_rvs_dt)

      # Create final list and save as .rds with timestamp
      final_mappings_list <- list(final_icd_dt = final_icd_dt, final_rvs_dt = final_rvs_dt)
      output_rds <- paste0(final_output_folder, "final_map_", year_to_process, suffix, latest_timestamp, ".rds")
      saveRDS(final_mappings_list, output_rds)

      # Save debugging CSVs
      output_icd_csv <- paste0(final_output_folder, "icd_", year_to_process, suffix, latest_timestamp, ".csv")
      output_rvs_csv <- paste0(final_output_folder, "rvs_", year_to_process, suffix, latest_timestamp, ".csv")

      fwrite(final_icd_dt, output_icd_csv)
      fwrite(final_rvs_dt, output_rvs_csv)

      # Move partial .rds files to the corresponding folder (cut instead of copy)
      file.rename(files, file.path(partial_output_folder, basename(files)))

      message("Final mappings saved to: ", output_rds)
      message("ICD mappings saved to: ", output_icd_csv)
      message("RVS mappings saved to: ", output_rvs_csv)
      message("Partial mapping files moved to: ", partial_output_folder)
    }
  }
}


Parallelization: TRUE 
[1] 2018


No files found for the specified year_to_load and suffix: _sampled_625_

No files found for the specified year_to_load and suffix: _sampled_125_

No files found for the specified year_to_load and suffix: _sampled_25_

No files found for the specified year_to_load and suffix: _sampled_5_

No files found for the specified year_to_load and suffix: _full_



Parallelization: TRUE 
[1] 2019


No files found for the specified year_to_load and suffix: _sampled_625_

No files found for the specified year_to_load and suffix: _sampled_125_

No files found for the specified year_to_load and suffix: _sampled_25_

No files found for the specified year_to_load and suffix: _sampled_5_

No files found for the specified year_to_load and suffix: _full_



Parallelization: TRUE 
[1] 2020


...

No files found for the specified year_to_load and suffix: _sampled_625_

No files found for the specified year_to_load and suffix: _sampled_125_

No files found for the specified year_to_load and suffix: _sampled_25_

No files found for the specified year_to_load and suffix: _sampled_5_

No files found for the specified year_to_load and suffix: _full_



Parallelization: TRUE 
[1] 2022


Final mappings saved to: ~/pids-drg-claims/data-cleaning/data/chkpts/chkpt_12_mapping/sampled_final_mappings/2022_sampled_625_2025-04-08 06:28:21.559632/final_map_2022_sampled_625_2025-04-08 06:28:21.559632.rds

ICD mappings saved to: ~/pids-drg-claims/data-cleaning/data/chkpts/chkpt_12_mapping/sampled_final_mappings/2022_sampled_625_2025-04-08 06:28:21.559632/icd_2022_sampled_625_2025-04-08 06:28:21.559632.csv

RVS mappings saved to: ~/pids-drg-claims/data-cleaning/data/chkpts/chkpt_12_mapping/sampled_final_mappings/2022_sampled_625_2025-04-08 06:28:21.559632/rvs_2022_sampled_625_2025-04-08 06:28:21.559632.csv

Partial mapping files moved to: ~/pids-drg-claims/data-cleaning/data/chkpts/chkpt_12_mapping/sampled_rds_files/2022_sampled_625_2025-04-08 06:28:21.559632/

No files found for the specified year_to_load and suffix: _sampled_125_

No files found for the specified year_to_load and suffix: _sampled_25_

No files found for the specified year_to_load and suffix: _sampled_5_

No file

Parallelization: TRUE 
[1] 2023


No files found for the specified year_to_load and suffix: _sampled_625_

No files found for the specified year_to_load and suffix: _sampled_125_

No files found for the specified year_to_load and suffix: _sampled_25_

No files found for the specified year_to_load and suffix: _sampled_5_

No files found for the specified year_to_load and suffix: _full_

